In [ ]:
from util_class import transform_image

import os
from PIL import Image

import numpy as np
from tqdm.notebook import tqdm

np.set_printoptions(
    threshold=np.inf,      # don't truncate elements
    linewidth=np.inf       # don't wrap lines (extends to full screen width)
)

NUM_EXISTING_LABEL = 10

label_encoding = {}

expression_folder = [x[0] for x in os.walk("../data/extracted_images")]
expression_folder = expression_folder[1:]

for i, expression_path in tqdm(enumerate(expression_folder, start=NUM_EXISTING_LABEL), total=len(expression_folder)):
    folder_name = os.path.basename(expression_path)

    label_encoding[folder_name] = i

    with os.scandir(expression_path) as images:
        image_files = [entry for entry in images if entry.is_file()]

        x = np.ndarray((len(image_files), 28, 28))

        for j, file in tqdm(enumerate(image_files), total=len(image_files)):
            img = transform_image(Image.open(file))
            x[j] = img

        with open(f"../data/transformed_images/{folder_name}_x.npy", "wb") as file:
            np.save(file, x)

        y = np.full((len(image_files),), fill_value=i)

        with open(f"../data/transformed_images/{folder_name}_y.npy", "wb") as file:
            np.save(file, y)

  0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/14294 [00:00<?, ?it/s]

  0%|          | 0/14355 [00:00<?, ?it/s]

  0%|          | 0/25112 [00:00<?, ?it/s]

  0%|          | 0/4483 [00:00<?, ?it/s]

  0%|          | 0/13104 [00:00<?, ?it/s]

  0%|          | 0/199 [00:00<?, ?it/s]

  0%|          | 0/3251 [00:00<?, ?it/s]

In [13]:
expression_folder = [x[0] for x in os.walk("../data/extracted_images")][1:]

X_train = np.ndarray((0, 28, 28))
Y_train = np.ndarray((0,))

X_test = np.ndarray((0, 28, 28))
Y_test = np.ndarray((0,))

# Training size to testing size.
split_ratio = 6/7

for expression_path in tqdm(expression_folder):
    folder_name = os.path.basename(expression_path)
    
    curr_X = np.load(f"../data/transformed_images/{folder_name}_x.npy")
    curr_Y = np.load(f"../data/transformed_images/{folder_name}_y.npy")

    split_idx = int(len(curr_X) * split_ratio)
    curr_X_train, curr_X_test = np.split(curr_X, [split_idx])
    curr_Y_train, curr_Y_test = np.split(curr_Y, [split_idx])

    X_train = np.concatenate((X_train, curr_X_train))
    X_test = np.concatenate((X_test, curr_X_test))

    Y_train = np.concatenate((Y_train, curr_Y_train))
    Y_test = np.concatenate((Y_test, curr_Y_test))


train_perm = np.random.permutation(len(X_train))
test_perm = np.random.permutation(len(X_test))

X_train = X_train[train_perm]
X_test = X_test[test_perm]

Y_train = Y_train[train_perm]
Y_test = Y_test[test_perm]

np.save("../data/processed_data/X_train.npy", X_train)
np.save("../data/processed_data/X_test.npy", X_test)
np.save("../data/processed_data/Y_train.npy", Y_train)
np.save("../data/processed_data/Y_test.npy", Y_test)

  0%|          | 0/7 [00:00<?, ?it/s]

In [11]:
Y = np.load("../data/processed_data/Y_train.npy")

print(Y.shape, Y.dtype)

(64110,) float64
